## Build SVD

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
movies = pd.read_csv("../data/ml-latest-small/movies.csv")
ratings = pd.read_csv("../data/ml-latest-small/ratings.csv")

In [12]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_movie_matrix.shape

(610, 9724)

In [13]:
user_movie_matrix_norm = user_movie_matrix.subtract(user_movie_matrix.mean(axis=1), axis=0)
user_movie_matrix_norm = user_movie_matrix_norm.fillna(0)

In [15]:
similarity = cosine_similarity(user_movie_matrix_norm.T)

In [16]:
svd = TruncatedSVD(n_components=50, random_state=42)

matrix_reduced = svd.fit_transform(user_movie_matrix_norm)

In [17]:
matrix_reconstructed = np.dot(matrix_reduced, svd.components_)

In [18]:
predicted_ratings = pd.DataFrame(
    matrix_reconstructed,
    index=user_movie_matrix_norm.index,
    columns=user_movie_matrix_norm.columns
)

In [19]:
def recommend_movies_svd(user_id, n=10):
    user_ratings = predicted_ratings.loc[user_id]
    
    # Remove already watched
    watched = ratings[ratings["userId"] == user_id]["movieId"]
    recommendations = user_ratings.drop(watched)
    
    # Sort
    top_movies = recommendations.sort_values(ascending=False).head(n)
    
    return movies[movies["movieId"].isin(top_movies.index)][["title"]]

In [21]:
recommend_movies_svd(1)

,title
686,Rear Window (1954)
792,"Sound of Music, The (1965)"
868,Wallace & Gromit: The Wrong Trousers (1993)
945,Dead Poets Society (1989)
1545,"Little Mermaid, The (1989)"
2078,"Sixth Sense, The (1999)"
2110,"Christmas Story, A (1983)"
2996,Snatch (2000)
3194,Shrek (2001)
3638,"Lord of the Rings: The Fellowship of the Ring,..."
